In [3]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import os 
from shapely.geometry import Point
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.patches import FancyArrowPatch
import matplotlib.image as mpimg


root = fr"C:\Users\eunic\Dropbox\sa_fires"
main_dir =  fr"{root}/proj_bureaucrats_farms"
int_path = fr"{main_dir}/data_output/intermediate"


# Panel A

In [5]:
# Import grid and states
grid = gpd.read_file( fr"{root}/proj_downwind/data_output/intermediate/1-grid-generation.shp" )
polygons = gpd.read_file( fr'{root}/data/input/ac_boundaries/Constituencies_Boundaries_Post_2008.shp' )


# Selecting States
sel_states = ["PUNJAB", "HARYANA",  "UTTAR PRADESH",  "BIHAR"]
sel_states_shp = polygons[ polygons.STATE_UT.isin( sel_states ) ]
fire_icon = mpimg.imread( f'{root}/proj_downwind/tex/paper/figures/fire.png')

# Acs selected
acs = gpd.read_file(fr'{int_path}/_0_2_3_ACs_right_shapefile.shp')
sel_polygon = acs.query('ac_uq_id == 780').copy()

In [ ]:
# Assuming sel_states_shp and polygons are already loaded
# Select the district with dist_id == 156
selected_acs = acs.query('ac_uq_id == 780').copy()

# Calculate the total bounds of sel_states_shp and add a margin
xmin, ymin, xmax, ymax = sel_states_shp.total_bounds
x_margin = 1.0  # 1 degree added to each side
y_margin = 1.0

# Set up the main plot
fig, ax = plt.subplots(figsize=(30, 30))

# Plot sel_states_shp with green fill and no edge
sel_states_shp.plot(ax=ax, color='#AFE1AF', edgecolor=None)

# Plot polygons with only boundaries in blue
polygons.boundary.plot(ax=ax, color='#336ece')

# Set the axis limits with added margins
ax.set_xlim(xmin - x_margin, xmax + x_margin)
ax.set_ylim(ymin - y_margin, ymax + y_margin)

# Add other plot customizations if needed
ax.axis('off')  # Turn off the axis

# Create an inset axis for the zoomed-in district
inset_ax = fig.add_axes([0.6, 0.45, 0.3, 0.3])  # [left, bottom, width, height] in figure coordinates

# Plot the selected district in the inset
polygons.plot(ax=inset_ax, color='#AFE1AF', edgecolor='black')

# Optionally, plot the surrounding polygons with boundaries in the inset (if needed)
polygons.boundary.plot(ax=inset_ax, color='#336ece', linewidth=0.5)

# adding the grid
grid.plot(ax=inset_ax, facecolor='none', edgecolor='black', linewidth=0.1)

# Set limits for the inset axis based on the bounds of the selected district with a margin
xmin_inset, ymin_inset, xmax_inset, ymax_inset = selected_acs.total_bounds
x_inset_margin = 0.1  # Adjust margin as needed for better visibility
y_inset_margin = 0.02

inset_ax.set_xlim(xmin_inset - x_inset_margin, xmax_inset + x_inset_margin)
inset_ax.set_ylim(ymin_inset - y_inset_margin, ymax_inset + y_inset_margin)

# Hide axis for the inset plot
inset_ax.axis('off')

# Add a red square around the inset plot
rect = Rectangle((0, 0), 1, 1, transform=inset_ax.transAxes,
                 color='red', fill=False, lw=5, zorder=9)
inset_ax.add_patch(rect)

# Calculate the centroid of the selected district
centroid = selected_acs.geometry.centroid.iloc[0]



# Add an arrow from the centroid to the midpoint of the inset's bottom line
arrow = FancyArrowPatch((centroid.x, centroid.y), (centroid.x + 5.4, centroid.y + 2.2),
                        transform=ax.transData,
                        color='red', arrowstyle='->', lw=3, zorder=10)
ax.add_patch(arrow)

# Save the code
fig.savefig( fr"{main_dir}/tex/paper/figures/map_grids.pdf", 
            format='pdf', dpi=300, bbox_inches='tight')

# Show the plot
plt.show()


# Panel C

In [6]:
from exactextract import exact_extract
import os
root = fr'C:/Users/eunic/Dropbox/sa_fires'
# GPW v4 population *density* raster (persons per km^2), 30 arc-sec (~1km), EPSG:4326
pop_tif = os.path.join(
    root,
    "data",
    "input",
    "population_density",
    "sedac",
    "gpw-v4-population-density-rev11_2015_30_sec_tif",
    "gpw_v4_population_density_rev11_2015_30_sec.tif",
)

# Confirm that the file exists
if not os.path.isfile(pop_tif):
    raise FileNotFoundError(f"Raster not found:\n{pop_tif}")


# Area-weighted mean density within each grid cell.
# exact_extract weights each pixel by its fractional overlap with the grid
# and skips nodata (water), so partial-coverage cells are handled correctly.
grid["mean_pop_density"] = exact_extract(pop_tif, grid, ["mean"], output="pandas")["mean"].values

# Grid area in km^2 (project to the metric CRS used elsewhere in this notebook)
grid["area_km2"] = grid.to_crs(7755).area / 1e6

# Population per grid = density (persons/km^2) * area (km^2)
grid["population"] = grid["mean_pop_density"].fillna(0) * grid["area_km2"]

population = grid  # GeoDataFrame with a per-grid `population` column
population[["unq_s__", "mean_pop_density", "area_km2", "population"]].head()

Exception: Unhandled raster datatype

In [4]:
# Area of Grids
grid = gpd.read_file(fr"{int_path}/1-grid-generation.shp")
acs = gpd.read_file(fr'{int_path}/_0_2_2_final_ac_rural_clean_from_finalver15_only_polygon.shp')
acs['geometry'] = acs['geometry'].buffer(0)  # Fix invalid geometries

grid['area_grid'] = grid.to_crs(7755).area
inter_grid_ac = gpd.overlay(grid, acs)

# Intersection of Grids
inter_grid_ac['inter_grid_area'] = inter_grid_ac.to_crs(7755).area
inter_grid_ac['frac_area_grid'] = inter_grid_ac['inter_grid_area'] * 100 / inter_grid_ac['area_grid']
inter_grid_ac = inter_grid_ac.reset_index(drop = True)

# Selecting grids with highest share of area in an ACs
idxmax = inter_grid_ac.groupby('unq_s__')['frac_area_grid'].idxmax()
max_area_grid = inter_grid_ac.loc[ idxmax ].copy() \
                        .reset_index(drop = True)

# Selecting the grids for ACs 912
grid_idx = max_area_grid.query('ac_uq_id == 912')['unq_s__'].tolist()

# Selecting grids
selected_grids = grid[grid.unq_s__.isin(grid_idx)].copy()

# Selecting the AC value
selected_polygon = acs.query(' ac_uq_id == 912')

c:\Users\eunic\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: C:\Users\eunic\Dropbox\sa_fires/proj_bureaucrats_farms/data_output/intermediate/_0_2_2_final_ac_rural_clean_from_finalver15_only_polygon.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(
c:\Users\eunic\AppData\Local\Programs\Python\Python312\Lib\site-packages\geopandas\tools\overlay.py:357: UserWarning: `keep_geom_type=True` in overlay resulted in 8 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)


In [5]:
import matplotlib.image as mpimg

# Select the grid where unq_s__ == 7289
selected_grid = selected_grids[selected_grids["unq_s__"] == 7289]

# Get the centroid of the selected grid
centroid = selected_grid.geometry.centroid.iloc[0]

# Define the arrow direction: NE to SW (45-degree slope)
arrow_dx = 1  # Moving left (west)
arrow_dy = -1  # Moving down (south)

# Compute the perpendicular line equation: y = mx + c
m_arrow = arrow_dy / arrow_dx  # Slope of the arrow (-1)
m_perpendicular = 1  # Perpendicular slope (inverse and negation of arrow slope)
c_perpendicular = centroid.y - m_perpendicular * centroid.x

# Identify grids below the perpendicular line
def is_below_perpendicular(geometry):
    x, y = geometry.centroid.x, geometry.centroid.y
    return y < (m_perpendicular * x + c_perpendicular)

selected_grids["below_line"] = selected_grids.geometry.apply(is_below_perpendicular)
filter1 = selected_grids.unq_s__.isin([7295, 7283])
selected_grids.loc[filter1, 'below_line'] = False

# Population per grid (in thousands, rounded)
selected_grids["population"] = selected_grids["unq_s__"].map(
    population.set_index("unq_s__")["population"])
selected_grids["population"] = round(selected_grids["population"]/1000,0)

# Load fire image
fire_img_path = fr"{int_path}/fire.png"
fire_img = mpimg.imread(fire_img_path)


# Plotting
fig, ax = plt.subplots(figsize=(8, 6))
selected_grids.boundary.plot(ax=ax, color="black", linewidth=0.5)
# Two flat colors: light red under the line (downwind), light blue above (upwind)
selected_grids[selected_grids["below_line"]].plot(ax=ax, color="lightcoral", alpha=0.6)
selected_grids[~selected_grids["below_line"]].plot(ax=ax, color="lightblue", alpha=0.6)

# Overlay the fire image at the centroid
# New extent for a smaller fire image
fire_size_factor = 0.005
img_extent = [
    centroid.x - fire_size_factor, centroid.x + fire_size_factor,
    centroid.y - fire_size_factor, centroid.y + fire_size_factor
]
ax.imshow(fire_img, extent=img_extent, aspect='auto', zorder=9)

# Draw the arrow
ax.arrow(
    centroid.x-0.015, centroid.y+0.015, arrow_dx * 0.01, arrow_dy * 0.01,  # Adjust arrow length
    head_width=0.005, head_length=0.005, fc="black", ec="black", zorder=11
)

selected_polygon.plot(ax=ax, color="none", edgecolor="black", linewidth=1.5, label="Selected Polygon")

ax.axis("off")  # Remove axis for better visualization

# Population label at each grid centroid, in dark letters
for _, row in selected_grids.iterrows():
    centroid = row.geometry.centroid
    ax.text(centroid.x, centroid.y, f"{row['population']:.0f}",
            fontsize=8, ha="center", va="center", color="black", fontweight="bold")

# Fraction: population under the line (light red numerator) over
# population above the line (light blue denominator)
num = selected_grids.loc[selected_grids["below_line"], "population"].sum()
den = selected_grids.loc[~selected_grids["below_line"], "population"].sum()
ax.text(0.45, 0.15, f"{num:,.0f}", transform=ax.transAxes, ha="center", va="top",
        fontsize=24, color="lightcoral", clip_on=False)
ax.plot([0.4, 0.5], [0.08, 0.08], transform=ax.transAxes,
        color="black", lw=1.5, clip_on=False)
ax.text(0.45, 0.055, f"{den:,.0f}", transform=ax.transAxes, ha="center", va="top",
        fontsize=24, color="lightblue", clip_on=False)

save_path = fr"{main_dir}/tex/paper/figures/panel_A_downwind.png"  # Change to your desired path
plt.savefig(save_path, bbox_inches="tight", dpi=300)

plt.show()


C:\Users\eunic\AppData\Local\Temp\ipykernel_20920\4087810762.py:7: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroid = selected_grid.geometry.centroid.iloc[0]


NameError: name 'population' is not defined